# 06 — Demand Modeling (ABM)

**Parsimonious Agent-Based Demand Model.**  
Synthetic BEV agents travel each road segment. Those below range-anxiety threshold charge.
Aggregate sessions → charger count per segment. Peak scenario used for sizing.

Inputs: interurban road network (IMD) + EV fleet projection 2027 + seasonal multipliers  
Output: `demand_per_segment.csv`

## Data Inputs
- `data/processed/interurban_roads.parquet` — road segments with IMD traffic counts
- `data/processed/ev_projection_2027.csv` — SARIMA forecast (mandatory: 2,498,159)

## Data Output
- `data/processed/demand_per_segment.csv`
  - Columns: `segment_id, route_segment, daily_bev_traffic_2027, n_chargers_needed, is_tent, seasonal_multiplier`

In [ ]:
import sys
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path

sys.path.append('..')
from src.constants import (
    EV_FLEET_2027, EV_FLEET_DEMAND_BASE, EV_PENETRATION_RATE,
    BEV_FRACTION, TOTAL_VEHICLE_FLEET,
    CHARGING_PROBABILITY, AVG_CHARGE_DURATION_HOURS, EFFECTIVE_OPERATING_HOURS,
    SOC_MEAN, SOC_STD, RANGE_ANXIETY_THRESHOLD, EFFECTIVE_RANGE_KM,
    MIN_CHARGERS_TENT, MIN_CHARGERS_STANDARD,
    MAX_CHARGERS_HIGH_TRAFFIC, MAX_CHARGERS_STANDARD, HIGH_TRAFFIC_IMD_THRESHOLD,
    MEDITERRANEAN_ROADS, ATLANTIC_ROADS,
)
from src.abm_demand import (
    get_seasonal_multiplier,
    compute_daily_bev_flow,
    compute_chargers_for_segment,
)

DATA_DIR = Path('../data/processed')
print('✅ Imports OK')
print(f'   EV fleet 2027 (mandatory):  {EV_FLEET_2027:,}')
print(f'   Demand base (conservative): {EV_FLEET_DEMAND_BASE:,}')
print(f'   EV penetration rate:        {EV_PENETRATION_RATE:.4f} ({EV_PENETRATION_RATE*100:.2f}%)')
print(f'   BEV fraction:               {BEV_FRACTION:.0%}')
print(f'   Charging probability (B1):  {CHARGING_PROBABILITY:.0%}')
print(f'   Session duration (B2):      {AVG_CHARGE_DURATION_HOURS*60:.0f} min')
print(f'   Operating hours (B3):       {EFFECTIVE_OPERATING_HOURS} hrs/day')
print(f'   Effective range (A2):       {EFFECTIVE_RANGE_KM} km')
print(f'   Range anxiety threshold:    {RANGE_ANXIETY_THRESHOLD:.0%} SOC → {RANGE_ANXIETY_THRESHOLD*EFFECTIVE_RANGE_KM:.0f} km buffer')

## Step 1: Load inputs

Load the interurban road network (with IMD traffic) and verify the mandatory EV fleet projection value.

In [ ]:
# Load road segments with IMD traffic
roads = gpd.read_parquet(DATA_DIR / 'interurban_roads.parquet')
print(f'📊 Road segments loaded: {len(roads):,}')
print(f'   Columns: {list(roads.columns)}')

# Load EV projection — verify mandatory value
ev_proj = pd.read_csv(DATA_DIR / 'ev_projection_2027.csv')
total_ev = int(ev_proj[ev_proj['type'] == 'forecast']['cumulative_ev_fleet'].max())
print(f'\n📈 EV projection 2027 (SARIMA): {total_ev:,}')
assert total_ev == EV_FLEET_2027, (
    f'EV projection mismatch: got {total_ev}, expected {EV_FLEET_2027}'
)
print('✅ EV projection matches mandatory value')

# Quick summary
print(f'\n🚗 IMD statistics:')
print(f'   Segments with IMD: {roads["imd_total"].notna().sum():,}')
print(f'   IMD range: {roads["imd_total"].min():.0f} – {roads["imd_total"].max():.0f} vehicles/day')
print(f'   IMD median: {roads["imd_total"].median():.0f}')
print(f'   TEN-T segments: {roads["is_tent"].sum():,}')

## Step 2: ABM Demand Model — Daily BEV Flow

**Formula:** `daily_bev_traffic_2027 = IMD_total × EV_penetration_rate × BEV_fraction`

- `EV_penetration_rate = 2,000,000 / 35,000,000 ≈ 5.71%` (E1/E3)
- `BEV_fraction = 60%` — PHEVs use ICE on long trips (A4)

Each unit of daily BEV flow = one synthetic weighted agent in the ABM corridor.

In [ ]:
# Fill missing IMD values with road-level median, then global median
if 'Carretera' in roads.columns:
    roads['imd_total'] = roads.groupby('Carretera')['imd_total'].transform(
        lambda x: x.fillna(x.median())
    )
roads['imd_total'] = roads['imd_total'].fillna(roads['imd_total'].median())

# Vectorised daily BEV flow
roads['daily_bev_traffic_2027'] = roads['imd_total'].apply(compute_daily_bev_flow).round(1)

print(f'⚡ Daily BEV flow computed:')
print(f'   Total daily BEV vehicle-segments: {roads["daily_bev_traffic_2027"].sum():,.0f}')
print(f'   Mean per segment: {roads["daily_bev_traffic_2027"].mean():.1f}')
print(f'   Max per segment:  {roads["daily_bev_traffic_2027"].max():.1f} (high-traffic corridor)')

# Seasonal multiplier — size for peak demand (ABM worst-case scenario)
roads['seasonal_multiplier'] = roads['Carretera'].apply(
    lambda name: get_seasonal_multiplier(name, scenario='peak')
)

print(f'\n🌞 Seasonal multiplier distribution:')
labels = {1.0: 'Standard', 1.5: 'Atlantic peak ×1.5',
          2.0: 'Mediterranean shoulder ×2.0', 2.5: 'Mediterranean peak ×2.5'}
for mult, count in roads['seasonal_multiplier'].value_counts().sort_index().items():
    print(f'   {labels.get(mult, f"×{mult}")}: {count:,} segments')

## Step 3: Charger Sizing (ABM Behavioral Model)

**Formula:**
```
daily_demand_hours = daily_bev_flow × seasonal_mult × CHARGING_PROBABILITY × AVG_CHARGE_DURATION_HOURS
n_chargers = ceil(daily_demand_hours / EFFECTIVE_OPERATING_HOURS)
```
Clamped to AFIR compliance: `[MIN_CHARGERS_TENT=4, MAX_CHARGERS_HIGH_TRAFFIC=12]` on TEN-T,
`[MIN_CHARGERS_STANDARD=2, MAX_CHARGERS_STANDARD=8]` on other roads.

In [ ]:
# Determine TEN-T tier (core vs comprehensive)
if 'tent_tier' not in roads.columns:
    roads['tent_tier'] = roads['is_tent'].map({True: 'core', False: 'none'})

# Compute charger count using ABM behavioral model
roads['n_chargers_needed'] = roads.apply(
    lambda r: compute_chargers_for_segment(
        daily_bev_flow=r['daily_bev_traffic_2027'],
        is_tent_core=(r['tent_tier'] == 'core'),
        is_tent_comp=(r['tent_tier'] == 'comprehensive'),
        imd_total=r['imd_total'],
        seasonal_multiplier=r['seasonal_multiplier'],
    ),
    axis=1
)

print('⚡ ABM charger sizing results:')
print(roads['n_chargers_needed'].value_counts().sort_index().to_string())
print(f'\n   Min: {roads["n_chargers_needed"].min()}  Max: {roads["n_chargers_needed"].max()}')
print(f'   Mean: {roads["n_chargers_needed"].mean():.2f}')
print(f'   Segments requiring 4+ (TEN-T/high demand): {(roads["n_chargers_needed"] >= 4).sum():,}')

## Step 4: Validation & Save

In [ ]:
# Assemble output columns
has_seg_id = 'segment_id' in roads.columns

demand_out = pd.DataFrame({
    'segment_id': roads['segment_id'] if has_seg_id else roads.index,
    'route_segment': roads['Carretera'],
    'daily_bev_traffic_2027': roads['daily_bev_traffic_2027'],
    'n_chargers_needed': roads['n_chargers_needed'],
    'is_tent': roads['is_tent'],
    'seasonal_multiplier': roads['seasonal_multiplier'],
})

# --- Validation ---
assert demand_out['daily_bev_traffic_2027'].notna().all(), 'NaN in daily_bev_traffic_2027'
assert demand_out['n_chargers_needed'].between(
    MIN_CHARGERS_STANDARD, MAX_CHARGERS_HIGH_TRAFFIC
).all(), f'n_chargers_needed out of [{MIN_CHARGERS_STANDARD}, {MAX_CHARGERS_HIGH_TRAFFIC}]'
assert len(demand_out) > 0, 'Empty output'

print(f'✅ Validation passed')
print(f'   Rows: {len(demand_out):,}')
print(f'   All n_chargers_needed in [{MIN_CHARGERS_STANDARD}, {MAX_CHARGERS_HIGH_TRAFFIC}]')
print(f'   No NaN in daily_bev_traffic_2027')

# --- Save ---
out_path = DATA_DIR / 'demand_per_segment.csv'
demand_out.to_csv(out_path, index=False)
print(f'\n💾 Saved → {out_path}')
print(f'   Columns: {list(demand_out.columns)}')
demand_out.head(3)